# RetinaGuard APTOS evaluation

This notebook creates a fixed stratified split, calibrates the abstention threshold on validation data, and reports locked test-set results against two baselines. The evaluation does not call OpenRouter; a separate live demonstration below does.

Before running, accept the APTOS 2019 competition rules on Kaggle and add a Colab secret named `KAGGLE_API_TOKEN`. The live demonstration also needs an `OPENROUTER_API_KEY` secret.

In [ ]:
from pathlib import Path
import subprocess
import sys

repo = Path('/content/PE6201-End-of-Course-Project')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/MauriceChang1027/PE6201-End-of-Course-Project', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements-colab.txt')], check=True)

In [ ]:
import kagglehub

data_dir = Path('/content/aptos2019')
competition_path = Path(kagglehub.competition_download(
    'aptos2019-blindness-detection',
    output_dir=str(data_dir),
))
competition_path

In [ ]:
labels_csv = next(data_dir.rglob('train.csv'))
images_dir = next(path for path in data_dir.rglob('train_images') if path.is_dir())
results_dir = repo / 'evaluation' / 'results'
print('Labels:', labels_csv)
print('Images:', images_dir)

In [ ]:
subprocess.run([
    sys.executable, '-m', 'scripts.evaluate_aptos',
    '--labels-csv', str(labels_csv),
    '--images-dir', str(images_dir),
    '--output-dir', str(results_dir),
    '--batch-size', '32',
    '--seed', '6201',
    '--target-sensitivity', '0.90',
    '--max-abstention-rate', '0.15',
], cwd=repo, check=True)

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv(results_dir / 'summary_metrics.csv'))
display(pd.read_csv(results_dir / 'quality_gate_summary.csv'))
display(Image(filename=results_dir / 'confusion_matrix_referable_raw.png'))
display(Image(filename=results_dir / 'threshold_calibration.png'))
display(Image(filename=results_dir / 'baseline_comparison.png'))

## Recording-ready Grade 3 demonstration

After the evaluation cells finish, run the next two cells once before recording. They reuse the verified Grade 3 image and evaluation report already in this runtime; no local image upload or second dataset download is needed. During recording, show the prepared outputs or rerun only these two cells. The OpenRouter key is read from Colab Secrets and is never printed. This selected case demonstrates the workflow, not population-level accuracy. The five-image evidence includes a Grade 2 false negative.

In [ ]:
import hashlib
import json
import sys
from PIL import Image as PILImage
import matplotlib.pyplot as plt

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from retinaguard import RetinaGuardPipeline
from retinaguard.config import load_confidence_setting

evidence = json.loads((repo / 'evidence' / 'five_image_acceptance.json').read_text(encoding='utf-8'))
demo_case = next(case for case in evidence['cases'] if case['ground_truth_grade'] == 3)
demo_image_path = images_dir / f"{demo_case['image_id']}.png"
if hashlib.sha256(demo_image_path.read_bytes()).hexdigest() != demo_case['image_sha256']:
    raise ValueError('Demo image does not match the recorded APTOS evidence.')

confidence_setting = load_confidence_setting(results_dir / 'evaluation_report.json')
pipeline = RetinaGuardPipeline(confidence_threshold=confidence_setting.threshold)
analysis = pipeline.analyse(demo_image_path)
triage = analysis.triage
if triage is None or triage.abstained:
    raise RuntimeError('The selected image did not produce an accepted vision result.')
explanation = pipeline.explain(demo_image_path, analysis)

figure, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(PILImage.open(demo_image_path).convert('RGB'))
axes[0].set_title('APTOS image')
axes[1].imshow(explanation.heatmap_image)
axes[1].set_title('Grad-CAM heatmap')
axes[2].imshow(explanation.overlay)
axes[2].set_title('Attribution overlay')
for axis in axes:
    axis.axis('off')
plt.tight_layout()
plt.show()

print(f"Locked test label: Grade {demo_case['ground_truth_grade']}")
print(f'Vision result: Grade {triage.prediction.grade} - {triage.prediction.label}')
print(f'Confidence: {triage.prediction.confidence:.1%}; threshold: {confidence_setting.threshold:.0%}')
print(f'Urgency: {triage.urgency}; action: {triage.action}')
print('Clinical validation: not established; clinician review required.')

In [ ]:
from google.colab import userdata
from retinaguard import OpenRouterClient

referral = OpenRouterClient(userdata.get('OPENROUTER_API_KEY')).generate_safe_referral(triage, 'DEMO-001')
print('Live OpenRouter draft for licensed clinician review only')
print(referral.text)
print(f'LLM validation passed: {referral.validation_passed}')
print(f'Deterministic fallback used: {referral.used_fallback}')

In [ ]:
from google.colab import files
import shutil

archive = shutil.make_archive('/content/retinaguard_evaluation_results', 'zip', results_dir)
files.download(archive)